# NorthCUA-Nano — computer-use agent on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/melvin/northcua-nano-computer-use-ee73/colab/NorthCUA-Nano-computer-use-agent.ipynb)

[`beipo-ai/NorthCUA-Nano`](https://huggingface.co/beipo-ai/NorthCUA-Nano) is a **9B dense** computer-use vision-language model (Apache 2.0): post-trained on [`Qwen/Qwen3.5-9B`](https://huggingface.co/Qwen/Qwen3.5-9B) (`Qwen3_5ForConditionalGeneration`, Hub pipeline `image-text-to-text`). It takes a user request, a screenshot, and interaction history, then emits reasoning plus native tool calls for a desktop or browser runtime. GUI coordinates use the official **`normalized_1000`** contract — `x` and `y` each sit in **[0, 1000]** relative to the screenshot, not raw pixels. The Hub checkpoint is four BF16 shards, about **18.84 GB** on disk / **9.41B** parameters.

**Why this model.** Most computer-use agents are closed or huge. NorthCUA-Nano is the compact open member of a 9B / 35B / 397B family: one policy that sees the screen and speaks the official [NorthCUA Harness](https://github.com/beipo-ai/NorthCUA) tool language (the old [`NorthCUA-Harness`](https://github.com/china-qijizhifeng/NorthCUA-Harness) repo redirects there). The technical report lists **78.90% OSWorld-Verified** and **67.16% WebArena-Verified** after RL; the card says those numbers are not tied to this exact export.

This notebook is a **safe single-step policy demo**, not live desktop control. It renders synthetic PIL windows, fetches one official NorthCUA demo frame, prints the raw model output, parses `<tool_call>` blocks, and draws any predicted click / drag / move. Nothing is sent to a window manager, browser, or shell.

**Licence:** [Apache 2.0](https://www.apache.org/licenses/LICENSE-2.0) (same as the Qwen3.5 base). Stock Transformers classes — no `trust_remote_code`.

**Runtime / VRAM assumption:** *Runtime > Change runtime type > T4 GPU*. A free T4 has about **15 GB** of VRAM, **no native bfloat16 tensor cores**, and **no FlashAttention-2**. The card's snippet loads `dtype=torch.bfloat16` on a machine with enough aggregate accelerator memory. That is the wrong type and too many bytes for 16 GB: BF16 weights alone are ~18.8 GB, and even an unquantized float16 copy is ~9–10 GB before vision activations. This notebook therefore loads **4-bit NF4 + float16 compute** via bitsandbytes from the start (about **6–7 GB** packed). Screenshots are capped so the long side is **1280 px**, and `max_new_tokens` stays at **768** (the public Space uses 3072). Expected peak: **~8–12 GB** — an estimate from the packed weights plus one 1280 px image, not a measured Colab number yet. Attention is **SDPA** (eager if SDPA refuses Qwen3.5's hybrid / linear-attention stack). Do not install `flash-attn`. The checkpoint records **transformers 5.20.2**; this notebook pins that release. Qwen3.5 is not in 4.x.

**Colab T4 pitfalls this notebook plans for.** Qwen3-VL-family vision `conv3d` has raised `CUDNN_STATUS_SUBLIBRARY_VERSION_MISMATCH` on Colab under float16; the generate helper disables cuDNN (`torch.backends.cudnn.enabled = False`) before every `generate`, matching the NetraV4 notebook. Newer Transformers releases have broken some VLM paths in this repo (Mage-VL-Cowork needed a 4.57 pin). NorthCUA-Nano needs a 5.x that actually knows `Qwen3_5ForConditionalGeneration` — **5.20.2**, the export version.

**Not in this notebook:** the official Harness runtime (NexAU, Playwright, a real desktop/VNC), OSWorld rollouts, FlashAttention-2, BF16, or the community [`mradermacher/NorthCUA-Nano-GGUF`](https://huggingface.co/mradermacher/NorthCUA-Nano-GGUF) path. Prefer Transformers; GGUF is only a last resort if 4-bit load fails on Colab.

**Links:** [model card](https://huggingface.co/beipo-ai/NorthCUA-Nano) · [NorthCUA Harness](https://github.com/beipo-ai/NorthCUA) · [project](https://beipo.top/blog/northcua/) · [report](https://arxiv.org/abs/2611.02102) · [demo Space](https://huggingface.co/spaces/hugging-apps/northcua-nano) · [announcement](https://x.com/PtVHVQ2FPQ51994/status/2117229952231702676)


## 1. Install Transformers 5.20.2 and bitsandbytes

The [model card](https://huggingface.co/beipo-ai/NorthCUA-Nano) loads stock `AutoProcessor` + `Qwen3_5ForConditionalGeneration`. The export records `transformers_version: 5.20.2`. Qwen3.5 landed in Transformers 5.2; **4.x cannot load this checkpoint**. The public [Space](https://huggingface.co/spaces/hugging-apps/northcua-nano) asks for `transformers>=5.20.2`. This notebook pins **exactly 5.20.2** so a later Colab preinstall cannot silently break the vision processor the way 5.21 broke Mage-VL-Cowork.

`Qwen3VLProcessor` imports torchvision even for still images. Use **`%pip`** (not `python -I -m pip`). Isolated mode drops Colab's site-packages and breaks the preinstalled `torch`. Keep Colab's torch. Do **not** install `flash-attn`. Do **not** install `flash-linear-attention` — Qwen3.5's linear-attention layers have a reference PyTorch path (same choice as the Xiaomi-OCR-0 notebook). `bitsandbytes` is the T4 load path, not a fallback: unquantized weights do not fit.


In [ ]:
%pip install -q "transformers==5.20.2" "accelerate>=1.1.0" "bitsandbytes>=0.45.0" torchvision pillow

# Checkpoint / card: transformers 5.20.2 (Qwen3_5ForConditionalGeneration).
# 4.x does not know qwen3_5. Pin exact — newer 5.x has broken VLM processors
# in this repo (Mage-VL-Cowork on 5.21).
# bitsandbytes is the T4 path: ~18.8 GB bf16 does not fit ~15 GB.
# torchvision is required by Qwen3VLProcessor even for still images.
# Use %pip, not python -I -m pip — isolated mode drops Colab site-packages.
# Do NOT replace Colab's torch. Do NOT install flash-attn or flash-linear-attention.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled Transformers. The 5.20.2 pin is invisible until the process restarts.


## 2. Check the GPU and print T4 caveats

The next cell must see a CUDA GPU. It prints `torch.cuda.is_available()`, the device name, and free VRAM, then stops if you are on CPU.


In [ ]:
import gc

import torch
import transformers
from packaging.version import Version

_tf_ver = Version(transformers.__version__.split("+")[0])
assert _tf_ver == Version("5.20.2"), (
    f"transformers {transformers.__version__} is loaded; NorthCUA-Nano needs "
    "exactly 5.20.2 (checkpoint / Qwen3_5ForConditionalGeneration). "
    "Run the install cell, then Runtime > Restart session."
)

print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2 and no native bfloat16 tensor cores.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

DEVICE = "cuda"
major, minor = torch.cuda.get_device_capability()
try:
    NATIVE_BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
except TypeError:
    NATIVE_BF16 = major >= 8
free, total = torch.cuda.mem_get_info()
DTYPE = torch.float16  # 4-bit compute dtype; T4 has no native bf16

print("transformers", transformers.__version__, "| torch", torch.__version__)
print("device:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"VRAM free: {free / 1024**3:.1f} GB of {total / 1024**3:.1f} GB")
print("native bf16:", NATIVE_BF16, "| FlashAttention-2: off (sdpa / eager only)")
print("load path: 4-bit NF4 + float16 compute (bf16 ~18.8 GB does not fit a T4)")
print("screenshot cap: long side 1280 px | max_new_tokens: 768")


def print_vram(label=""):
    suffix = f"  ({label})" if label else ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"VRAM allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB | "
        f"peak: {peak:.2f} GB | free: {free_now / 1024**3:.1f}/{total_now / 1024**3:.1f} GB{suffix}"
    )


## 3. Load NorthCUA-Nano in 4-bit NF4

`AutoProcessor` + `Qwen3_5ForConditionalGeneration`, matching the [card's Transformers snippet](https://huggingface.co/beipo-ai/NorthCUA-Nano#quickstart). No `trust_remote_code`. The first run downloads about **19 GB**, then packs the weights to 4-bit (~6–7 GB on the GPU).

The system prompt and tool schemas below are the public [Space](https://huggingface.co/spaces/hugging-apps/northcua-nano) port of the official Harness templates (`northcua/agents/desktop/prompt/templates/*.j2` and `northcua_runtime_shared/desktop_surface/action_contracts.py`), with `normalized_1000` filled in. Do not skip them, and **do** pass `tools=TOOLS` into `apply_chat_template` — that is how the native chat template emits the `<tools>` / `<tool_call><function=…>` contract the model was trained on.

Official OSWorld sampling in the report is `temperature=0.7`, `top_p=0.95`, `top_k=40`, 1,024 output tokens. Demos here stay **greedy** and **768** new tokens so a T4 finishes.


In [ ]:
import json
import math
import re
from pathlib import Path

from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3_5ForConditionalGeneration

MODEL_ID = "beipo-ai/NorthCUA-Nano"
MAX_SIDE = 1280
MAX_NEW_TOKENS = 768
ATTN = None
model = None
last_error = None

# Ported from hugging-apps/northcua-nano (official Harness system + tools,
# coordinate policy = normalized_1000). Do not invent a different schema.
SYSTEM_TEMPLATE = """You are ComputerUsingAgent, a computer-using agent whose job is to complete the user's query directly on the desktop.

Goal:
- Complete the user's query accurately, safely, and efficiently.
- Use the current desktop state, retained tool-call trace, and retained screenshots to decide the next best step.

Runtime context:
- Desktop environment: {desktop_environment}

Task completion contract:
- At the start of the task, classify the user's request privately:
  - answer_task: the user wants a natural-language answer only.
  - state_task: the user wants an application, browser, page, setting, or UI to be left in a specific verifiable state.
  - artifact_task: the user wants a file, spreadsheet, document, chart, or other artifact created or modified.
- Do not treat a natural-language answer as completion for a state_task or artifact_task.
- Only report completion after verification. If verification fails or is uncertain, continue correcting the state.

Reasoning protocol for every step:
1. Review the retained tool-call trace and screenshots in chronological order.
2. Summarize the current screen state and whether it matches the expected state.
3. Decide the next best action: what to do, which control, how you localize it on the latest screenshot, and why.
4. Prefer the smallest reliable action that makes progress. Do not repeat ineffective actions.
5. Prefer the target application's GUI for GUI tasks. Use read-only checks only to verify ambiguous final state.
6. After every GUI action, use the next screenshot as the authoritative confirmation.

Response rules:
- Use only the native tool definitions supplied by the API request.
- Do not restate tool schemas in natural language.
- When an action is required, emit native tool calls and also include a short non-empty user-facing text block that explains current progress, the action you are about to take, and why.
- Do not return tool calls alone.
- When the task is complete, failed, or cannot be completed reliably, do not emit tool calls. Reply with plain text only.
- If the environment itself makes the task infeasible, start the plain-text reply with `[INFEASIBLE]` and explain the missing requirement.
- Do not ask the user for clarification.

Screenshot semantics:
- Use the newest available screenshot as the authoritative state.
- If the current user message includes a screenshot, that image is the latest full desktop state for this turn and should be used for coordinate grounding.
- Coordinate policy for click, double_click, move, drag path points, and optional scroll x/y: Return x,y normalized to [0,1000] relative to the freshest grounding screenshot.
- Coordinate origin is the top-left corner of the full grounding screenshot; `x` increases to the right and `y` increases downward.
- For 0-1000 coordinates, use `(0, 0)` for the top-left of the screenshot and `(1000, 1000)` for the bottom-right, regardless of the screenshot's pixel resolution.
- Scroll `delta_x` and `delta_y` are wheel deltas, not screen coordinates; only optional scroll `x`/`y` follow the coordinate policy.

Current screenshot resolution: {width} x {height} pixels."""

PERSISTENT_USER = """User query:
{task}

Long-lived instruction:
Complete the user query accurately. Use the retained tool-call trace and screenshots to avoid repeating ineffective actions. Stop with a plain-text status report when the task is finished or clearly blocked.
If the task cannot be performed because the environment lacks a required app, permission, display, file, account, network resource, or platform capability, stop with plain text starting with `[INFEASIBLE]` and name the missing requirement."""

OBSERVATION_USER = """Current observation:
- Current retained image count: 1
- The image below is the latest full desktop state for this turn. Use it as the primary grounding image for the next action.
- Any tool coordinates must refer to the freshest clean grounding image, in normalized [0, 1000] space.

Use this observation and the retained tool-call trace to determine current progress and what the next best action should be."""

_BUTTON = {"type": "string", "default": "left"}
_XY = {
    "x": {"type": "number", "description": "Horizontal coordinate in normalized [0, 1000] space."},
    "y": {"type": "number", "description": "Vertical coordinate in normalized [0, 1000] space."},
}
_SHOT = " Returns plain-text feedback plus the post-action screenshot. This notebook never executes the call."


def _fn(name, desc, props, required):
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": desc,
            "parameters": {
                "type": "object",
                "properties": props,
                "required": required,
                "additionalProperties": False,
            },
        },
    }


TOOLS = [
    _fn(
        "note",
        "Record important text for later steps without interacting with the desktop.",
        {
            "key": {"type": "string", "default": "", "description": "Stable memory key."},
            "text": {"type": "string", "description": "The exact information to preserve."},
            "reason": {"type": "string", "default": "", "description": "Why this note matters."},
            "kind": {
                "type": "string",
                "enum": ["fact", "mistake", "precaution", "plan", "constraint", "decision", "other"],
                "default": "fact",
            },
        },
        ["text"],
    ),
    _fn(
        "screenshot",
        "Capture a fresh desktop screenshot without sending input." + _SHOT,
        {"all_screens": {"type": "boolean", "default": False}},
        [],
    ),
    _fn(
        "click",
        "Click a desktop coordinate in normalized [0, 1000] space." + _SHOT,
        {**_XY, "button": {**_BUTTON, "description": 'Mouse button: "left", "right", or "middle".'}},
        ["x", "y"],
    ),
    _fn(
        "double_click",
        "Double-click a desktop coordinate in normalized [0, 1000] space." + _SHOT,
        {**_XY, "button": {**_BUTTON, "description": 'Mouse button: "left", "right", or "middle".'}},
        ["x", "y"],
    ),
    _fn(
        "drag",
        "Drag through an ordered list of desktop coordinates in normalized [0, 1000] space. At least two points." + _SHOT,
        {
            "path": {
                "type": "array",
                "minItems": 2,
                "description": "Ordered drag path.",
                "items": {
                    "type": "object",
                    "properties": _XY,
                    "required": ["x", "y"],
                    "additionalProperties": False,
                },
            },
            "button": {**_BUTTON, "description": "Mouse button to hold during drag."},
        },
        ["path"],
    ),
    _fn("move", "Move the mouse pointer to a desktop coordinate in normalized [0, 1000] space." + _SHOT, dict(_XY), ["x", "y"]),
    _fn(
        "scroll",
        "Scroll at the current pointer or at a normalized [0, 1000] coordinate. Positive delta_y scrolls up." + _SHOT,
        {
            "x": {"type": "number", "description": "Optional horizontal coordinate."},
            "y": {"type": "number", "description": "Optional vertical coordinate."},
            "delta_x": {"type": "integer", "default": 0},
            "delta_y": {"type": "integer", "default": 0, "description": "Positive scrolls up, negative down."},
        },
        [],
    ),
    _fn(
        "type",
        "Literal text to enter into the currently focused editable control. Focus the field first. Prefer keypress for shortcuts." + _SHOT,
        {
            "text": {"type": "string", "description": "Literal text to type or paste."},
            "target_kind": {
                "type": "string",
                "enum": ["auto", "editable", "terminal"],
                "default": "auto",
            },
        },
        ["text"],
    ),
    _fn(
        "keypress",
        'Press one key or a key chord. Provide exactly one of `key` or `keys`. Example: keys=["ctrl+l", "tab"].' + _SHOT,
        {
            "key": {"type": "string", "description": "Single key. Omit when using keys."},
            "keys": {"type": "array", "items": {"type": "string"}, "description": "Ordered press units. Omit when using key."},
        },
        [],
    ),
    _fn(
        "wait",
        "Wait without sending input, then capture a fresh screenshot." + _SHOT,
        {"ms": {"type": "integer", "default": 1000, "description": "Wait duration in milliseconds."}},
        [],
    ),
]

print("Loading processor…")
processor = AutoProcessor.from_pretrained(MODEL_ID)
EOS_IDS = [i for i in (processor.tokenizer.eos_token_id, processor.tokenizer.pad_token_id) if i is not None]

quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print("Loading NorthCUA-Nano in 4-bit NF4 (first run ~19 GB download)…")
for attn in ("sdpa", "eager"):
    try:
        print(f"trying 4-bit + {attn}…")
        model = Qwen3_5ForConditionalGeneration.from_pretrained(
            MODEL_ID,
            quantization_config=quant,
            attn_implementation=attn,
            device_map={"": 0},
            low_cpu_mem_usage=True,
        ).eval()
        ATTN = attn
        break
    except Exception as exc:
        last_error = exc
        print(f"{attn} 4-bit load failed:", type(exc).__name__, exc)
        try:
            del model
        except NameError:
            pass
        model = None
        gc.collect()
        torch.cuda.empty_cache()

if model is None:
    raise RuntimeError(
        "Could not load beipo-ai/NorthCUA-Nano in 4-bit on this GPU. "
        "If Transformers + bitsandbytes cannot load Qwen3.5-VL here, the community "
        "GGUF is mradermacher/NorthCUA-Nano-GGUF — this notebook does not install it."
    ) from last_error

DEVICE = next(model.parameters()).device
n_params = sum(p.numel() for p in model.parameters()) / 1e9
print("class:", type(model).__name__)
print("processor:", type(processor).__name__)
print("attention:", ATTN, "| quantization: 4-bit nf4 / float16 compute")
print(f"parameters: {n_params:.2f}B")
print("eos ids:", EOS_IDS)
print_vram("after load")


## 4. Helpers — harness prompt, tool-call parse, and click drawing

The native chat template (and the Space) use this XML shape, **not** a JSON blob inside `<tool_call>`:

```
<think>…</think>
short user-facing text
<tool_call>
<function=click>
<parameter=x>
742
</parameter>
<parameter=y>
188
</parameter>
</function>
</tool_call>
```

Pixel conversion (card / Harness): `x_px = round(x_norm / 1000 * width)`, same for `y`. `(500, 500)` is the screenshot centre. `(0, 0)` is top-left; `(1000, 1000)` is bottom-right.

**Colab cuDNN workaround.** Some Colab T4 images fail Qwen3-VL-family vision patch embedding during generate with `CUDNN_STATUS_SUBLIBRARY_VERSION_MISMATCH`. `policy_step` disables cuDNN before every generate so all three demos take the same path. Float16 compute + SDPA/eager is unchanged.


In [ ]:
import io
import time

import requests
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont

UA = {
    "User-Agent": (
        "donvito-notebooks/1.0 (educational Colab demo; "
        "https://github.com/donvito/notebooks)"
    )
}

TOOL_CALL_RE = re.compile(r"<tool_call>(.*?)(?:</tool_call>|$)", re.S)
FUNC_RE = re.compile(r"<function=([^>\s]+)>(.*?)(?:</function>|$)", re.S)
PARAM_RE = re.compile(r"<parameter=([^>\s]+)>\s*(.*?)\s*</parameter>", re.S)


def _coerce(value):
    try:
        return json.loads(value)
    except Exception:
        return value


def split_output(raw, thinking=True):
    """Split a generation into (reasoning, visible text, tool calls). Space parser."""
    raw = raw.split("<|im_end|>", 1)[0]
    reasoning, body = "", raw
    if "</think>" in raw:
        reasoning, body = raw.split("</think>", 1)
    elif thinking and "<think>" in raw:
        reasoning, body = raw, ""
    reasoning = reasoning.replace("<think>", "").strip()
    calls = []
    for block in TOOL_CALL_RE.findall(body):
        for name, inner in FUNC_RE.findall(block):
            calls.append({"name": name, "arguments": {k: _coerce(v) for k, v in PARAM_RE.findall(inner)}})
    text = FUNC_RE.sub("", TOOL_CALL_RE.sub("", body)).strip()
    return reasoning, text, calls


def to_pixels(x_norm, y_norm, width, height):
    """Card: (500, 500) is centre. x_px = round(x / 1000 * width)."""
    return int(round(float(x_norm) / 1000.0 * width)), int(round(float(y_norm) / 1000.0 * height))


def downscale(image, max_side=MAX_SIDE):
    image = image.convert("RGB")
    longest = max(image.size)
    if longest > max_side:
        scale = max_side / longest
        image = image.resize(
            (max(1, int(image.width * scale)), max(1, int(image.height * scale))),
            Image.Resampling.LANCZOS,
        )
    return image


def fetch_image(url, max_side=MAX_SIDE):
    for attempt in range(4):
        response = requests.get(url, headers=UA, timeout=120)
        if response.status_code not in (429, 500, 502, 503, 504) or attempt == 3:
            response.raise_for_status()
            return downscale(Image.open(io.BytesIO(response.content)), max_side)
        retry_after = response.headers.get("Retry-After", "")
        time.sleep(int(retry_after) if retry_after.isdigit() else 2 ** (attempt + 1))
    raise RuntimeError(f"could not download {url}")


def _font(size=18):
    for path in (
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
    ):
        if Path(path).exists():
            return ImageFont.truetype(path, size)
    return ImageFont.load_default()


def _text_size(draw, text, font):
    box = draw.textbbox((0, 0), text, font=font)
    return box[2] - box[0], box[3] - box[1]


def draw_window(canvas, box, title, fill=(246, 247, 249)):
    x0, y0, x1, y1 = box
    draw = ImageDraw.Draw(canvas)
    draw.rounded_rectangle(box, radius=10, fill=fill, outline=(60, 64, 72), width=2)
    draw.rectangle((x0, y0, x1, y0 + 36), fill=(47, 52, 61))
    draw.ellipse((x0 + 12, y0 + 12, x0 + 24, y0 + 24), fill=(232, 80, 80))
    draw.ellipse((x0 + 30, y0 + 12, x0 + 42, y0 + 24), fill=(240, 190, 60))
    draw.ellipse((x0 + 48, y0 + 12, x0 + 60, y0 + 24), fill=(80, 180, 90))
    draw.text((x0 + 72, y0 + 8), title, fill=(240, 242, 245), font=_font(16))
    return canvas


def render_contact_form(width=1280, height=720):
    """Fake Acme Portal contact form. Submit is large and high-contrast."""
    img = Image.new("RGB", (width, height), (38, 70, 110))
    draw = ImageDraw.Draw(img)
    draw.rectangle((0, height - 40, width, height), fill=(32, 36, 42))
    draw.text((16, height - 30), "Activities    Files    Firefox", fill=(220, 224, 230), font=_font(15))
    window = (180, 70, 1100, 640)
    draw_window(img, window, "Acme Portal — Contact")
    x0, y0, x1, y1 = window
    draw.text((x0 + 36, y0 + 56), "New ticket", fill=(30, 34, 40), font=_font(28))
    draw.text((x0 + 36, y0 + 96), "Fill the form, then click Submit.", fill=(80, 86, 96), font=_font(16))

    def field(label, value, top):
        draw.text((x0 + 36, top), label, fill=(50, 54, 62), font=_font(15))
        draw.rounded_rectangle((x0 + 36, top + 24, x1 - 36, top + 64), radius=6, fill=(255, 255, 255), outline=(170, 176, 186), width=2)
        draw.text((x0 + 48, top + 34), value, fill=(40, 44, 52), font=_font(16))

    field("Name", "Ada Lovelace", y0 + 140)
    field("Email", "ada@example.com", y0 + 220)
    draw.text((x0 + 36, y0 + 300), "Message", fill=(50, 54, 62), font=_font(15))
    draw.rounded_rectangle((x0 + 36, y0 + 324, x1 - 36, y0 + 430), radius=6, fill=(255, 255, 255), outline=(170, 176, 186), width=2)
    draw.text((x0 + 48, y0 + 340), "Please provision the staging VM.", fill=(40, 44, 52), font=_font(16))

    cancel = (x1 - 360, y1 - 90, x1 - 210, y1 - 42)
    submit = (x1 - 190, y1 - 90, x1 - 36, y1 - 42)
    draw.rounded_rectangle(cancel, radius=8, fill=(226, 229, 234), outline=(140, 146, 156), width=2)
    draw.rounded_rectangle(submit, radius=8, fill=(26, 115, 232), outline=(16, 80, 170), width=2)
    cw, ch = _text_size(draw, "Cancel", _font(18))
    sw, sh = _text_size(draw, "Submit", _font(18))
    draw.text(((cancel[0] + cancel[2] - cw) / 2, (cancel[1] + cancel[3] - ch) / 2), "Cancel", fill=(50, 54, 62), font=_font(18))
    draw.text(((submit[0] + submit[2] - sw) / 2, (submit[1] + submit[3] - sh) / 2), "Submit", fill=(255, 255, 255), font=_font(18))
    return img, {"submit": submit, "cancel": cancel, "window": window}


def render_browser_search(width=1280, height=720):
    """Fake Firefox search page with a focused query box and a Search button."""
    img = Image.new("RGB", (width, height), (56, 62, 72))
    draw = ImageDraw.Draw(img)
    draw.rectangle((0, 0, width, 48), fill=(32, 36, 42))
    draw.text((16, 12), "Firefox  —  search.example", fill=(220, 224, 230), font=_font(16))
    draw.rectangle((0, height - 36, width, height), fill=(32, 36, 42))
    draw.text((16, height - 28), "https://search.example", fill=(180, 186, 196), font=_font(14))
    window = (160, 80, 1120, 620)
    draw_window(img, window, "search.example — Search")
    x0, y0, x1, y1 = window
    draw.text((x0 + 48, y0 + 70), "Find docs", fill=(30, 34, 40), font=_font(32))
    box = (x0 + 48, y0 + 140, x1 - 220, y0 + 200)
    search = (x1 - 200, y0 + 140, x1 - 48, y0 + 200)
    draw.rounded_rectangle(box, radius=8, fill=(255, 255, 255), outline=(26, 115, 232), width=3)
    draw.text((box[0] + 16, box[1] + 16), "|", fill=(26, 115, 232), font=_font(22))
    draw.rounded_rectangle(search, radius=8, fill=(26, 115, 232), outline=(16, 80, 170), width=2)
    sw, sh = _text_size(draw, "Search", _font(18))
    draw.text(((search[0] + search[2] - sw) / 2, (search[1] + search[3] - sh) / 2), "Search", fill=(255, 255, 255), font=_font(18))
    draw.text((x0 + 48, y0 + 230), "The query box is focused (blue outline). Type, then click Search.", fill=(80, 86, 96), font=_font(16))
    return img, {"query": box, "search": search, "window": window}


def draw_actions(image, calls):
    """Draw click / double_click / move / drag / scroll points. Space visualiser, PIL-safe."""
    img = image.convert("RGB").copy()
    width, height = img.size
    draw = ImageDraw.Draw(img)
    radius = max(8, int(min(width, height) * 0.012))
    lw = max(2, radius // 4)
    font = _font(max(14, int(radius * 1.3)))
    colors = [(255, 45, 85), (0, 122, 255), (52, 199, 89), (255, 149, 0)]

    def label(xy, txt, color):
        x, y = xy
        tw, th = _text_size(draw, txt, font)
        lx = min(max(x + radius + 4, 8), width - tw - 16)
        ly = min(max(y - radius - th - 8, 8), height - th - 16)
        draw.rectangle((lx - 4, ly - 2, lx + tw + 4, ly + th + 6), fill=color)
        draw.text((lx, ly), txt, fill=(255, 255, 255), font=font)

    def marker(xy, color, double=False):
        x, y = xy
        draw.ellipse((x - radius, y - radius, x + radius, y + radius), outline=color, width=lw)
        if double:
            draw.ellipse((x - radius * 1.6, y - radius * 1.6, x + radius * 1.6, y + radius * 1.6), outline=color, width=lw)
        draw.line((x - radius * 2, y, x + radius * 2, y), fill=color, width=lw)
        draw.line((x, y - radius * 2, x, y + radius * 2), fill=color, width=lw)

    pixels = []
    for i, call in enumerate(calls):
        args = call.get("arguments") or {}
        color = colors[i % len(colors)]
        tag = f"{i + 1}. {call.get('name')}"
        try:
            if call.get("name") in ("click", "double_click", "move") and "x" in args and "y" in args:
                xy = to_pixels(args["x"], args["y"], width, height)
                marker(xy, color, double=call.get("name") == "double_click")
                label(xy, f"{tag} [{args['x']}, {args['y']}] → px {xy}", color)
                pixels.append((call.get("name"), xy, args))
            elif call.get("name") == "drag" and isinstance(args.get("path"), list) and len(args["path"]) >= 2:
                pts = [to_pixels(p["x"], p["y"], width, height) for p in args["path"]]
                draw.line(pts, fill=color, width=lw)
                marker(pts[0], color)
                marker(pts[-1], color)
                label(pts[0], tag, color)
                pixels.append(("drag", pts, args))
            elif call.get("name") == "scroll" and "x" in args and "y" in args:
                xy = to_pixels(args["x"], args["y"], width, height)
                marker(xy, color)
                label(xy, f"{tag} dy={args.get('delta_y', 0)}", color)
                pixels.append(("scroll", xy, args))
        except Exception as exc:
            print("could not draw", call, exc)
    return img, pixels


def inside(box, xy):
    return box[0] <= xy[0] <= box[2] and box[1] <= xy[1] <= box[3]


def policy_step(task, image, history="", max_new_tokens=MAX_NEW_TOKENS):
    """One ComputerUsingAgent turn. Does not execute any tool."""
    # Colab T4 cuDNN version-mismatch workaround (vision patch embedding / conv3d).
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False

    image = downscale(image)
    width, height = image.size
    system = SYSTEM_TEMPLATE.format(desktop_environment="linux", width=width, height=height)
    user_text = PERSISTENT_USER.format(task=task.strip()) + "\n\n" + OBSERVATION_USER
    if history and history.strip():
        notes = [line.strip("-• ").strip() for line in history.strip().splitlines() if line.strip()]
        user_text += "\n\nAdditional observation notes:\n" + "\n".join(f"- Previous step: {n}" for n in notes)

    messages = [
        {"role": "system", "content": [{"type": "text", "text": system}]},
        {"role": "user", "content": [{"type": "text", "text": user_text}, {"type": "image", "image": image}]},
    ]
    inputs = processor.apply_chat_template(
        messages,
        tools=TOOLS,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=True,
    )
    inputs = inputs.to(DEVICE)
    for key, value in list(inputs.items()):
        if hasattr(value, "is_floating_point") and value.is_floating_point():
            inputs[key] = value.to(dtype=torch.float16)

    torch.cuda.reset_peak_memory_stats()
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=EOS_IDS,
            pad_token_id=EOS_IDS[-1],
        )
    raw = processor.decode(output[0, inputs["input_ids"].shape[-1] :], skip_special_tokens=False)
    reasoning, text, calls = split_output(raw, thinking=True)
    return {
        "raw": raw,
        "think": reasoning,
        "text": text,
        "calls": calls,
        "image": image,
        "size": image.size,
        "prompt_tokens": int(inputs["input_ids"].shape[-1]),
        "new_tokens": int(output.shape[-1] - inputs["input_ids"].shape[-1]),
    }


def show_prediction(result, hit_boxes=None):
    print("----- raw policy output -----")
    print(result["raw"])
    print("----- parsed -----")
    print("think:", (result["think"][:500] + "…") if len(result["think"]) > 500 else (result["think"] or "(none)"))
    print("user-facing:", result["text"] or "(none)")
    print("tool calls:", json.dumps(result["calls"], indent=2))
    print(f"tokens: prompt {result['prompt_tokens']} + new {result['new_tokens']}")
    print(f"screenshot: {result['size'][0]}x{result['size'][1]}  (0-1000 → pixels via x*W/1000)")
    annotated, pixels = draw_actions(result["image"], result["calls"])
    for kind, xy, args in pixels:
        print(f"  {kind} normalized { {k: args.get(k) for k in args} } → pixels {xy}")
        if hit_boxes and isinstance(xy, tuple) and len(xy) == 2:
            for label, box in hit_boxes.items():
                print(f"    inside {label}? {inside(box, xy)}  box={box}")
    if not pixels:
        print("  (no coordinate-bearing click/move/drag/scroll to draw)")
    display(annotated)
    print_vram("after generate")
    return result


## 5. Demo 1 — click Submit on a synthetic form

Safe single-step GUI demo. The screenshot is a PIL fake of a contact form, not a real desktop. The task is **click the Submit button**. The cell prints the raw output, parses any `click` in normalized `[0, 1000]`, converts to pixels, and draws the predicted point. Nothing is sent to a window manager.


In [ ]:
form_image, form_boxes = render_contact_form()
print("synthetic form", form_image.size, "| Submit box (pixels):", form_boxes["submit"])
display(form_image)

gui_result = policy_step("Click the Submit button on the contact form.", form_image)
show_prediction(gui_result, hit_boxes={"Submit": form_boxes["submit"], "Cancel": form_boxes["cancel"]})


## 6. Demo 2 — official NorthCUA demo frame (GeoGebra heart)

A real project screenshot: the Ruby Heart construction from the [NorthCUA Harness README](https://github.com/beipo-ai/NorthCUA) (Apache 2.0, © NorthCUA contributors). The Space uses the same frame with the task **Set the heart color to ruby red.**

Fetched from a stable GitHub raw URL. If the download fails, the cell falls back to a labelled PIL stand-in so the rest of the notebook still runs.


In [ ]:
HEART_URL = (
    "https://raw.githubusercontent.com/beipo-ai/NorthCUA/main/"
    "docs/assets/demos/ruby-heart.png"
)
try:
    heart = fetch_image(HEART_URL)
    print("official demo frame", heart.size, "| source: beipo-ai/NorthCUA docs/assets/demos (Apache 2.0)")
except Exception as exc:
    print("fetch failed, using PIL stand-in:", type(exc).__name__, exc)
    heart = Image.new("RGB", (1280, 720), (245, 247, 250))
    draw = ImageDraw.Draw(heart)
    draw_window(heart, (80, 40, 1200, 680), "GeoGebra — Ruby Heart (stand-in)")
    draw.ellipse((440, 180, 840, 560), outline=(180, 20, 40), width=8)
    draw.text((500, 350), "heart (stand-in)", fill=(180, 20, 40), font=_font(22))
    draw.rounded_rectangle((960, 80, 1180, 130), radius=6, fill=(26, 115, 232))
    draw.text((1000, 94), "Color", fill=(255, 255, 255), font=_font(16))

display(heart)
heart_result = policy_step("Set the heart color to ruby red.", heart)
show_prediction(heart_result)


## 7. Demo 3 — history plus a focused search box

Same style of synthetic Firefox window, but the query box is already focused (blue outline). The **history** string is passed through as the Space's "Previous steps" notes so the model sees a prior action, matching the card's "request + screenshot + history" interface.

The task is to **type `northcua` into the search box**. The interesting output is a `type` (and maybe `keypress` / `click` on Search). Predicted clicks are drawn; predicted `type` / `keypress` are printed only. Nothing is executed.


In [ ]:
browser_image, browser_boxes = render_browser_search()
print("synthetic browser", browser_image.size, "| Search box (pixels):", browser_boxes["search"])
display(browser_image)

browser_result = policy_step(
    "Type northcua into the search box, then submit the search.",
    browser_image,
    history="Clicked the query box so it is focused (blue outline).\nDid not click Search yet.",
)
show_prediction(
    browser_result,
    hit_boxes={"Search": browser_boxes["search"], "Query": browser_boxes["query"]},
)

print()
print("----- non-click tools (NOT executed) -----")
for call in browser_result["calls"]:
    if call.get("name") in {"type", "keypress", "note", "wait"}:
        print(call["name"], json.dumps(call.get("arguments", {}), ensure_ascii=False))
print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")


## 8. Caveats and cleanup

**What this notebook is for.** An inference-only T4 demo of NorthCUA-Nano: the official Harness system prompt + desktop tool schemas, `AutoProcessor.apply_chat_template(..., tools=TOOLS, enable_thinking=True)` + `Qwen3_5ForConditionalGeneration.generate`, and three single-step samples (synthetic click, official demo frame, history + type). Use it to see the `normalized_1000` contract and the allocated / peak VRAM.

**Not a desktop agent.** Screenshots are PIL fakes plus one official demo PNG. Predicted GUI actions are drawn, not replayed. Predicted `type` / `keypress` / `note` are printed. This notebook never runs the [NorthCUA Harness](https://github.com/beipo-ai/NorthCUA) itself.

**Prompt contract (from the Harness / Space).** Every acting turn is a `<think>` block, a short user-facing sentence, then one or more

`<tool_call><function=click><parameter=x>…</parameter><parameter=y>…</parameter></function></tool_call>`

blocks. Coordinates are normalized to **[0, 1000]** regardless of screenshot size (`x_px = round(x * width / 1000)`). Do not pass those numbers to a mouse API as pixels.

**Thinking / sampling.** The report's OSWorld-G protocol uses temperature 0.7 / top-p 0.95 / top-k 40 / 1,024 output tokens. This notebook is greedy and 768 new tokens so a free T4 finishes. Scores will not match the card.

**Transformers pin.** `transformers==5.20.2` (checkpoint). Qwen3.5 is 5.2+; 4.x will not load. If a later Colab image ships a broken 5.x processor, keep this pin.

**T4 / VRAM.** 4-bit NF4 + float16 compute from the start. Do not load bf16. Do not install FlashAttention-2. Screenshots are capped at 1280 px. cuDNN is disabled around `generate` because Qwen3-VL-family `conv3d` has mismatched cuDNN on some Colab T4 images. The last number below is `torch.cuda.max_memory_allocated` for this session.

**If 4-bit Transformers load fails on Colab.** Last-resort community GGUF: [`mradermacher/NorthCUA-Nano-GGUF`](https://huggingface.co/mradermacher/NorthCUA-Nano-GGUF). This notebook does not install llama.cpp. Prefer the Transformers path above.

**Responsible use.** The card is for research and development of computer-use agents. Verify consequential actions, isolate the execution environment, and keep credentials off the observed desktop. This notebook stays on synthetic / project-demo images for that reason.

**Licence.** Apache 2.0. See the checkpoint [NOTICE](https://huggingface.co/beipo-ai/NorthCUA-Nano/blob/main/NOTICE).

**Links again:** [model card](https://huggingface.co/beipo-ai/NorthCUA-Nano) · [Harness](https://github.com/beipo-ai/NorthCUA) · [project](https://beipo.top/blog/northcua/) · [report](https://arxiv.org/abs/2611.02102)

Free the GPU before you open another notebook:


In [ ]:
print_vram("before cleanup")
try:
    del model
except NameError:
    pass
try:
    del processor
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print_vram("after cleanup")
    print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print("NorthCUA-Nano released. Re-run the load cell if you want another prompt.")
